1. Test Tiếng Anh --> Tiếng Việt

In [ ]:
import pandas as pd
import requests
import json
from deep_translator import GoogleTranslator

TRANSLATE_URL = "https://dich-thuat-ai-nhom-05-dich-thuat-api-nhom5.hf.space/gradio_api/call/translate_logic"

# ===== 1. đọc file dạng RAW để không mất dòng =====
with open("test_english.csv", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

# clean cực mạnh
texts = []

for line in raw_lines:

    line = line.strip()

    if not line:
        texts.append("")   # giữ dòng rỗng
        continue

    texts.append(line)
results = []

for idx, text in enumerate(texts[:100]):

    print("=" * 60)
    print(idx, "SOURCE:", repr(text))

    payload = {
        "data": [
            text,
            "Anh -> Việt",
            True
        ]
    }

    try:
        r = requests.post(TRANSLATE_URL, json=payload)

        event_id = r.json().get("event_id")

        if not event_id:
            raise Exception("No event_id returned")

        url = f"{TRANSLATE_URL}/{event_id}"

        result = requests.get(url)

        translated = None

        for line in result.text.splitlines():
            if line.startswith("data:"):
                try:
                    translated = json.loads(
                        line.replace("data:", "").strip()
                    )[0]
                except:
                    translated = None
                break

        google = GoogleTranslator(
            source="en",
            target="vi"
        ).translate(text) if text else ""

        results.append({
            "source": text,
            "api_translation": translated,
            "google_translation": google,
            "match": translated == google
        })

    except Exception as e:

        print("ERROR:", e)

        results.append({
            "source": text,
            "api_translation": None,
            "google_translation": None,
            "match": False
        })

0 SOURCE: '\ufeffHello!'
1 SOURCE: 'Thank you very much.'
2 SOURCE: 'How are you?'
3 SOURCE: 'My name is John.'
4 SOURCE: 'What is this?'
5 SOURCE: 'I like apples.'
6 SOURCE: 'Where is the bus stop?'
7 SOURCE: 'It is a sunny day.'
8 SOURCE: 'Good morning.'
9 SOURCE: 'I am very hungry.'
10 SOURCE: 'She is reading a book.'
11 SOURCE: 'Please open the door.'
12 SOURCE: 'See you tomorrow.'
13 SOURCE: 'What time is it?'
14 SOURCE: 'I am a teacher.'
15 SOURCE: 'The car is fast.'
16 SOURCE: 'Can you speak English?'
17 SOURCE: 'This is my phone.'
18 SOURCE: 'I need some water.'
19 SOURCE: 'I have a question.'
20 SOURCE: 'Make yourself at home.'
21 SOURCE: 'It’s up to you.'
22 SOURCE: 'I’m running late.'
23 SOURCE: 'Long time no see.'
24 SOURCE: 'Give me a hand.'
25 SOURCE: 'Take care of yourself.'
26 SOURCE: "I don't think so."
27 SOURCE: 'Never mind.'
28 SOURCE: 'That sounds great.'
29 SOURCE: 'Keep in touch!'
30 SOURCE: 'I’m looking for my bag.'
31 SOURCE: "What's the matter?"
32 SOURCE: 'Pi

In [6]:
out = pd.DataFrame(results)

out.to_csv(
    "compareEnVi.csv",
    index=False,
    encoding="utf-8-sig"
)

print("DONE")

DONE


2. Test Tiếng Việt --> Tiếng Anh

In [1]:
import pandas as pd
import requests
import json
import time
from deep_translator import GoogleTranslator

TRANSLATE_URL = "https://dich-thuat-ai-nhom-05-dich-thuat-api-nhom5.hf.space/gradio_api/call/translate_logic"

# ===== 1. đọc RAW file =====
with open("test_vietnamese.csv", "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f.readlines()]

results = []

# ===== 2. loop =====
for idx, text in enumerate(texts[:100]):

    print("=" * 60)
    print(idx, "SOURCE:", repr(text))

    # skip None thật sự nhưng vẫn giữ dòng
    if text.lower() in ["nan", "null", "none", "undefined"]:
        results.append({
            "source": text,
            "api_translation": None,
            "google_translation": None,
            "match": False
        })
        continue

    payload = {
        "data": [
            text,
            "Việt -> Anh",
            True
        ]
    }

    try:
        # ===== POST =====
        r = requests.post(TRANSLATE_URL, json=payload)
        event_id = r.json().get("event_id")

        if not event_id:
            raise Exception("No event_id")

        url = f"{TRANSLATE_URL}/{event_id}"

        # ===== POLL cho tới khi complete =====
        translated = None

        for _ in range(20):

            res = requests.get(url)
            txt = res.text

            if "complete" in txt:

                for line in txt.splitlines():

                    if line.startswith("data:"):

                        try:
                            translated = json.loads(
                                line.replace("data:", "").strip()
                            )[0]
                        except:
                            translated = None

                        break

                break

            time.sleep(0.3)

        # ===== Google =====
        google = GoogleTranslator(
            source="vi",
            target="en"
        ).translate(text) if text else ""

        results.append({
            "source": text,
            "api_translation": translated,
            "google_translation": google,
            "match": translated == google
        })

        print("API   :", translated)
        print("GOOGLE:", google)

    except Exception as e:

        print("ERROR:", e)

        results.append({
            "source": text,
            "api_translation": None,
            "google_translation": None,
            "match": False
        })

# ===== 3. export =====
out = pd.DataFrame(results)
out.to_csv("compareViEn.csv", index=False, encoding="utf-8-sig")

print("DONE")

0 SOURCE: '\ufeffChào bạn.'
API   : Hey.
GOOGLE: ﻿Hello.
1 SOURCE: 'Bạn khỏe không?'
API   : How are you?
GOOGLE: How are you?
2 SOURCE: 'Tên tôi là Lan.'
API   : My name is Lan.
GOOGLE: My name is Lan.
3 SOURCE: 'Tôi đến từ Việt Nam.'
API   : I'm from Vietnam.
GOOGLE: I come from Vietnam.
4 SOURCE: 'Chúc một ngày tốt lành.'
API   : Have a nice day.
GOOGLE: Have a nice day.
5 SOURCE: 'Cái này là cái gì?'
API   : What is this?
GOOGLE: What is this?
6 SOURCE: 'Nó giá bao nhiêu?'
API   : How much does it cost?
GOOGLE: How much does it cost?
7 SOURCE: 'Tôi hiểu rồi.'
API   : I see.
GOOGLE: I got it.
8 SOURCE: 'Tôi không biết.'
API   : I do n't know.
GOOGLE: I do not know.
9 SOURCE: 'Làm ơn giúp tôi.'
API   : Please help me.
GOOGLE: Please help me.
10 SOURCE: 'Tôi thích xem phim.'
API   : I like to watch movies.
GOOGLE: I like watching movies.
11 SOURCE: 'Hôm nay trời đẹp.'
API   : It's a beautiful day.
GOOGLE: The weather is beautiful today.
12 SOURCE: 'Chúc mừng sinh nhật!'
API   : Happy 